# Enigma cracker on Google Colab

This notebook builds and runs the **Enigma I / M3 / M4 simulator and ciphertext-only cracker** from this repository.
The program is written in Rust and has no dependencies. Colab only needs to install the Rust compiler (about 1 minute) and build it (about 30 seconds).

**Run the cells from top to bottom** (Runtime → Run all works too; the long searches are at the end).

Good to know:
* The cracker runs on the **CPU only**; a GPU runtime does not help. A free Colab session usually has **2 CPU cores**, about half the speed of a 4-core laptop.
* Colab disconnects idle sessions and caps session length. For long searches, save the log to Google Drive (see step 6) or split the search into parts with `--part K/N` (see step 7).
* What it can break (see `README.md`): messages with **no plugboard or up to about 5 plugboard cables**, even when short. A full 10-cable plugboard on a short message is out of reach for any ciphertext-only attack; you would need a crib (a known word).

## 1. Install Rust

In [ ]:
!curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal > /dev/null
import os
os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]
!cargo --version

## 2. Get the code

If the GitHub repository is **public**, just run the cell.

If it is **private**, first create a GitHub token with read access to the repository (GitHub → Settings → Developer settings → Personal access tokens). Then add it as a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar) and allow this notebook to use it.

Change `BRANCH` to `main` once the code has been merged there.

In [ ]:
import os, subprocess

REPO = "levan07082009/ENIGMA"
BRANCH = "claude/eager-bohr-gr5a3j"

token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass  # no secret set: fine for a public repository

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.exists("/content/ENIGMA"):
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/ENIGMA"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        # Never print the URL: it may contain the token.
        raise RuntimeError("git clone failed (private repository without a GITHUB_TOKEN secret?)\n"
                           + r.stderr.replace(token or "\0", "***"))
%cd /content/ENIGMA
!git log --oneline -1

## 3. Build and self-test

The tests decrypt two real wartime messages, which checks that the machine model is exact:
* the 3-rotor *Operation Barbarossa* message (7 July 1941);
* the 4-rotor M4 message to U-264 (25 November 1942).

In [ ]:
!cargo build --release -q && cargo test --release -q 2>&1 | tail -3
!echo "CPU cores available: $(nproc)"

## 4. Try it: decrypt with known settings

Enigma is symmetric: the same settings encrypt and decrypt. With four rotors (a Greek wheel first), the tool acts as an M4.

In [ ]:
# The U-264 message (Enigma M4), broken by the M4 Message Breaking Project in 2006
!./target/release/enigma run --rotors Beta,II,IV,I --reflector B --rings AAAV --start VJNA \
    --plugs "AT BL DF GJ HM NW OP QY RZ VX" \
    NCZWVUSXPNYMINHZXMQXSFWXWLKJAHSHNMCOCCAKUQPMKCSMHKSEINJUSBLKIOSXCKUBHMLLXCSJUSRRDVKOHULXWCCBGVLIYX

## 5. Watch it crack a random key

The demo encrypts a German text with a random secret key and then cracks it without knowing the key. The key is drawn from the rotors and reflectors the crack searches, so restricting the search keeps the demo quick.

In [ ]:
# Enigma I, rotors I-V, 150 letters, 5 plugboard cables (about 2 minutes on 2 cores)
!./target/release/enigma demo --length 150 --plugs 5

In [ ]:
# Enigma M4, restricted to rotors I-III and the Beta wheel so it takes under a minute
!./target/release/enigma demo --machine m4 --rotors I,II,III --greek beta --reflector B --length 100 --plugs 3

## 6. Your ciphertext

Paste your message below; spaces and line breaks are ignored. The default is the unbroken U-534 message of 1 May 1945 (Enigma M4).

To keep the log if Colab disconnects, set `SAVE_TO_DRIVE = True`. That mounts your Google Drive and writes the logs to `MyDrive/enigma_logs/`.

In [ ]:
CIPHERTEXT = """
JCRSA JTGSJ EYEXY KKZZS HVUOC TRFRC RPFVY PLKPP LGRHV VBBTB RSXSW XGGTY TVKQN GSCHV GF
"""
SAVE_TO_DRIVE = False

import os
open("/content/ENIGMA/my_ciphertext.txt", "w").write(CIPHERTEXT)
LOG_DIR = "/content/ENIGMA/logs"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    LOG_DIR = "/content/drive/MyDrive/enigma_logs"
os.makedirs(LOG_DIR, exist_ok=True)
print("logs go to", LOG_DIR)
!./target/release/enigma analyze -f my_ciphertext.txt

## 7. Crack it

Pick the search that matches what you know about the message. The more you can pin down (machine, rotors, reflector), the faster and more reliable the search.

Typical times for a 72-letter message on a free 2-core Colab (roughly half that on 4 cores):

| Search | Command options | Time |
|---|---|---|
| Enigma I (army/air force), rotors I–V | *(defaults)* | ~1 min |
| M3, all 8 rotors, reflectors B and C | `--rotors all --reflector BC` | ~6 min |
| **M4, everything** (8 rotors, Beta+Gamma, thin B+C) | `--machine m4 --reflector BC` | **~1 hour** |

Useful options: `--lang de` or `--lang en` (default: both), `--max-plugs 0` (assume no plugboard), `--show 10` (print more candidates), `--threads N`.

In [ ]:
# Three-rotor machines, all rotors and both reflectors
!./target/release/enigma crack -f my_ciphertext.txt --rotors all --reflector BC --show 5 2>&1 | tee {LOG_DIR}/crack_m3.txt

### M4 (naval four-rotor Enigma)

The full M4 search is the long one. To split it across sessions, or across several Colab accounts or machines, set `PARTS` to how many pieces you want and `PART` to the piece this session runs (1..PARTS). Each part prints its own best results, and the overall answer is the best of all parts. With `PARTS = 1` it runs everything in one go.

In [ ]:
PART, PARTS = 1, 1

!./target/release/enigma crack -f my_ciphertext.txt --machine m4 --reflector BC --show 5 --part {PART}/{PARTS} 2>&1 | tee {LOG_DIR}/crack_m4_part{PART}of{PARTS}.txt

## 8. Reading the results

Each result shows the key in wartime terms and the decrypted text:
* **Walzenlage**: rotor order, left to right. An M4 order starts with the Greek wheel (Beta/Gamma).
* **Ringstellung**: ring settings.
* **Grundstellung**: start position.
* **Stecker**: plugboard pairs.

The printed key is *equivalent* to the original: it decrypts identically, but the left ring is always reported as `A`.

**Fitness** is 0 for random letters and 1 for typical text. A result marked *suspicious* has a high score only because a short text plus plugboard cables can be bent to fit letter statistics. **Only trust plaintext that actually reads as German or English words.** German military texts use `X` for a full stop, `Q` for `CH`, and spell out numbers (EINS, ZWO, DREI...).

If nothing readable comes out, the message probably uses a full plugboard. Next steps: find a *crib*. Run `enigma crib --crib WORD -f my_ciphertext.txt` to see where a guessed word can sit. Enigma never encrypts a letter to itself, which rules many positions out.

In [ ]:
!./target/release/enigma crib --crib KEINEBESONDERENEREIGNISSE -f my_ciphertext.txt